# 🇮🇩 Exploratory Data Analysis (EDA)
# Indonesian Data Profession Job Market

This notebook presents an exploratory analysis of the Indonesian data and machine learning job-posting dataset.

The goal is to unpack three focused questions about the current state of the market:

1. **Fresh Graduate Reality:** How welcoming is the industry towards entry-level candidates, and what is the minimum education standard?
2. **Technical Skill Landscape:** Which tools or programming languages are absolute must-haves?
3. **Salary & Remote Work Snapshot:** What compensation and work-location flexibility does the industry offer?

---

## TL;DR — Key Findings

> _From 1,247 data job postings in the cleaned Glints snapshot._

1. **The market is infrastructure-heavy** — ETL is the most-mentioned skill in this snapshot (**32.5%** of postings), alongside analytics, cloud, and BI tools.
2. **Jakarta dominates** — Jakarta accounts for **63.9%** of postings; Bandung (**10.8%**) and Surabaya (**6.7%**) are the next largest hubs.
3. **Remote work is limited** — **13.4%** of postings are classified as Remote, while Hybrid accounts for **23.7%**.
4. **Salary disclosure is complete in this snapshot** — all 1,247 cleaned rows contain a parsed salary value, so salary charts cover the full snapshot.
5. **The junior gap is visible** — only **5.6%** of postings have a zero-year minimum-experience requirement; most ask for at least one year.

---

## Business Questions Answered

- 🎓 *How junior-friendly is the Indonesian data job market?*
- 🛠️ *Which skills differentiate the four data roles?*
- 💰 *Do remote jobs pay more than on-site roles?*

Read on for the full analysis with charts.

## Setup and Data Loading

We load the cleaned master dataset produced by `clean.ipynb` and route every chart through :mod:`app.charts` so the notebook, the Streamlit dashboard, and the README previews share a single source of truth.

In [ ]:
import sys
from pathlib import Path

# Allow the notebook to import the package from `src/` and the app layer.
sys.path.insert(0, str(Path.cwd() / "src"))
sys.path.insert(0, str(Path.cwd()))

import pandas as pd

from app.charts import (
    plot_education_distribution,
    plot_experience_distribution,
    plot_remote_premium,
    plot_role_distribution,
    plot_salary_distribution,
    plot_salary_vs_experience,
    plot_skill_cooccurrence_heatmap,
    plot_skill_vs_salary,
    plot_top_companies,
    plot_top_skills_bar,
    plot_workplace_type_distribution,
)
from app.data import build_skill_columns

df = pd.read_csv("data/processed/master_dataset_eda.csv")
df, skill_cols = build_skill_columns(df)
print(f"loaded {len(df):,} rows · {len(skill_cols)} skill flags")

---

## The Fresh Graduate Reality in the Data Industry

This section uncovers how welcoming the Indonesian data industry is towards entry-level talent. We analyse two key metrics simultaneously:

1. **Experience Requirement (`exp_min_years`):** Are companies willing to accept candidates with 0 years of experience?
2. **Academic Standard (`job_education_clean`):** Is a bachelor's degree (S1) an absolute requirement, or is the industry open to vocational / diploma graduates?

In [ ]:
plot_experience_distribution(df).show()
plot_education_distribution(df).show()

### Conclusion: The Reality of Breaking Into the Data Industry

Looking at the combination of experience requirements and education standards above, the Indonesian data profession job market currently shows a relatively high *barrier to entry* for beginners.

On the experience side, only **5.6%** of postings in this snapshot have a zero-year minimum-experience requirement. Most ask for at least one year, which makes the entry-level segment comparatively narrow. Education requirements should be read as a separate signal because the cleaned snapshot groups free-text education into broad buckets.

For fresh graduates, this suggests a practical strategy: use internships, freelancing, or independent projects to demonstrate evidence of experience rather than treating the minimum-years field as the whole hiring decision.

---

## The Technical Skill Landscape (Tech Stack)

For a data practitioner, the *Tech Stack* (tools and programming languages) is one of the clearest signals available in a job posting.

This section dissects the technical keywords that appear most frequently in job descriptions. By mapping these competencies we can see whether the Indonesian market leans towards seeking *programmers* (Python / SQL) or *dashboard / reporting specialists* (Excel / Tableau / Power BI).

In [ ]:
plot_top_skills_bar(df, top_n=20).show()

The chart shows a varied technical stack rather than one universally dominant tool. **ETL** is the most-mentioned catalogue item in this snapshot, alongside analytics, cloud, and BI tools.

These counts measure keyword mentions, not verified requirements. A practical learning path still depends on the target role: analytics roles may emphasise SQL and BI, while engineering roles may combine ETL with cloud, orchestration, and storage tools.

---

## Salary Landscape & Work Flexibility

Now that we understand the technical and experience requirements, the next big question for job seekers is about compensation and work model.

This section breaks down the distribution of average salaries (`salary_avg`) offered in the market by experience level required. We will also look at the reality of work-location flexibility (`workplace_type`) to verify whether the popular myth that "data professionals can work from anywhere (Remote / WFH)" is actually reflected in Indonesian job postings today.

In [ ]:
plot_salary_distribution(df, by="exp_min_years").show()
plot_workplace_type_distribution(df).show()

### Conclusion: Compensation Curve and the Remote-Work Myth

Salary distributions differ across experience bands, but the spread is wide and the relationship should not be treated as a guarantee for an individual offer. Role, company, workplace type, and salary disclosure all affect the comparison.

On the work-model side, **On-site** postings are the majority (**62.9%**), followed by Hybrid (**23.7%**) and Remote (**13.4%**). Candidates looking for remote work should therefore expect a smaller search pool in this snapshot.

---

## Skill Co-occurrence — Which Skills Travel Together?

A single skill column tells you *demand*, but to understand the *shape* of the Indonesian data market you need to see which skills appear in the same posting. A job listing that asks for **Python + SQL** is a very different role from one that asks for **AWS + Spark**.

We use the **Jaccard similarity** — for two skills `A` and `B`:

```
Jaccard(A, B) = |A ∩ B| / |A ∪ B|
```

The resulting 12×12 heatmap highlights the natural skill clusters in the Indonesian market (data-engineering, BI/visualisation, ML, etc.).

In [ ]:
plot_skill_cooccurrence_heatmap(df, top_n=12).show()

**Reading the heatmap:** bright cells along the diagonal trivially equal 1.0 (a skill co-occurs perfectly with itself). Off-diagonal cells reveal the *real* structure:

- **Python ↔ SQL**, **Pandas ↔ NumPy**, and **Scikit-learn ↔ Machine Learning** show the strongest non-trivial overlap — these are the textbook analyst / scientist toolkits.
- **AWS / GCP / Azure** form a tight cluster with each other and with **Docker / Kubernetes** — the *cloud data engineering* stack.
- **Tableau / Power BI** cluster separately from the BI tooling above (closer to *Excel* than to Python) — pure-visualisation roles.

For a job-seeker, this chart answers the practical question *"what should I learn next?"* — pair your existing skills with whatever has the highest overlap with skills you already have.

---

## Salary vs Experience — the *Money Plot*

This is the chart every recruiter and candidate screenshots. Each dot is one job posting; the dashed line is an ordinary-least-squares fit across all roles.

> **Reading the chart:** the fitted line summarises the association between minimum experience and salary in this snapshot. It is descriptive, not a causal estimate or a promise that each additional year produces a fixed salary increase.

In [ ]:
plot_salary_vs_experience(df).show()

**What this tells you:** the association has a wide spread, meaning **what you do** matters alongside **how many years** the posting requests. The chart should support comparison, not be used to predict an individual offer.

---

## Which Skills Pay More? — Salary Premium per Skill

Counting job postings tells you what's in demand. To know **what's worth your time to learn**, we compute, for every skill, the difference in median salary between postings that *require* the skill and postings that *don't*.

> Positive values (green) mean the skill correlates with higher pay; negative values (red) mean the skill is associated with lower-paying postings.

In [ ]:
plot_skill_vs_salary(df, top_n=15).show()

**Take-away:** this chart compares medians for postings that mention a skill with those that do not. A premium is an association, not proof that the skill itself causes higher pay. Small groups can also be noisy, which is why the chart excludes groups with fewer than five disclosed-salary postings.

---

## Who is Hiring? — Top Companies & Remote Availability

The Indonesian data-job market is heavily concentrated: a small number of large employers drive a disproportionate share of postings. We close the analysis with two views:

1. **Top companies by posting count** — useful for understanding employer concentration.
2. **Top companies, split by on-site vs remote** — useful for identifying remote-friendly employers.

In [ ]:
plot_top_companies(df, top_n=15).show()
plot_remote_premium(df).show()
plot_role_distribution(df).show()

**Conclusion:** posting volume is concentrated among a small set of employers in this snapshot, but posting count is not the same as unique opportunity or employer size. Remote work remains the exception overall, so the company and workplace views should be read together rather than as standalone rankings.